# Simulation Calibration: Can the Digital Twin Correct Itself?

*Use case 2 of 3, about 20 minutes.*

Like many state-of-the-art simulators in the field, OpenDC uses a deterministic and static simulation strategy.
To stay accurate, OpenDT adds a Self-Calibrator, which measures the difference between simulation-predicted results and
actual telemetry, then continuously adjusts the simulation model.

In [ ]:
import numpy as np
import pandas as pd
from resources import *

check_setup()

## 1. The Power Model

We model CPU power draw adopting the OpenDC analytical formula $P(u) = P_{idle} + (P_{max} - P_{idle})(2u - u^r)$.
Here, $u$ is CPU utilization, $P_{idle}$ and $P_{max}$ represent idle and max power, and $r$ is the calibration
parameter.
For the SURF hosts, OpenDT uses $P_{idle}$ = 25 W, $P_{max}$ = 174 W, and $r$ = 4.

In [ ]:
u = np.linspace(0, 1, 101)  # CPU utilization, from 0 (idle) to 1 (full load)
P_IDLE, P_MAX = 25, 174  # watts

curves = {f"r = {r}": P_IDLE + (P_MAX - P_IDLE) * (2 * u - u**r) for r in [2, 4, 6, 9]}
ax = pd.DataFrame(curves, index=u * 100).plot()
ax.axhline(P_MAX, color="black", linewidth=0.8, linestyle="--", label="$P_{max}$")
ax.set(xlabel="CPU Utilization [%]", ylabel="Power Draw [W]", xlim=(0, 100), ylim=(0, None))
legend_above(ax)

In [ ]:
ask_question("2.1")

In [ ]:
give_solution("2.1", show=False)  # show=True shows the solution

In [ ]:
ask_question("2.2")

In [ ]:
give_solution("2.2", show=False)  # show=True shows the solution

## 2. Calibrating r with a Grid Search

The Self-Calibrator employs a grid search strategy over the power model's parameter space: it evaluates a set of
candidate parameter values by running short simulations and comparing the results against recent historical telemetry
data. The configuration yielding the lowest MAPE is selected and transmitted to the simulator for use in subsequent
predictions.
Here, we run one simulation of the full week per r, from 3 to 9, as in experiment E2 of the OpenDT paper (one to two
minutes).

In [ ]:
R_VALUES = [3, 4, 5, 6, 7, 8, 9]
sweep = simulate_many("r", R_VALUES, until=TRACE_END)  # one simulation of the full week per r


def error(run):
    """Error of a simulation at each minute [%]: |ground truth - prediction| / ground truth."""
    both = align(run)  # ground truth and prediction on the same 1-minute grid
    return (both["measured"] - both["simulated"]).abs() / both["measured"] * 100


errors = pd.DataFrame({r: error(run) for r, run in sweep.items()})  # one column per r
pd.DataFrame({"MAPE [%]": errors.mean()}).round(1)  # one row per r

In [ ]:
ask_question("2.3")

In [ ]:
give_solution("2.3", show=False)  # show=True shows the solution

## 3. The Best r per Day

In [ ]:
best_r_per_day = errors.resample("24h", origin=TRACE_START).mean().idxmin(axis=1)
best_r_per_day

In [ ]:
ask_question("2.4")

In [ ]:
give_solution("2.4", show=False)  # show=True shows the solution

## 4. The Self-Calibrating Digital Twin

In experiment E2, OpenDT twins one hour at a time; after each hour, the Self-Calibrator picks the r with the lowest MAPE
over the last hour, and the next hour uses it (in the live twin, the new r may arrive later).
OpenDC is deterministic, so we replay this from the simulations above, without new simulations.
The plot also shows the 10% error rate OpenDT should stay within (NFR1, community-accepted) and the MAPE of the
peer-reviewed simulator FootPrinter (7.86%).

In [ ]:
ask_question("2.5")

In [ ]:
hourly = errors.resample("1h").mean()  # MAPE of each r, per hour


def self_calibrate(hourly, picked):
    """MAPE per hour when each hour uses the r picked after the previous hour (r = 4 first)."""
    used = picked.shift(1, fill_value=4)
    return pd.Series([hourly.loc[hour, r] for hour, r in used.items()], index=hourly.index)


best_r = hourly.idxmin(axis=1)  # the r with the lowest MAPE in each hour
no_calibration = hourly[4]
with_calibration = self_calibrate(hourly, best_r)

for name, mape_per_hour in [("No Calibration", no_calibration), ("With Calibration", with_calibration)]:
    print(f"{name}: MAPE {mape_per_hour.mean():.1f}%, hours within 10%: {(mape_per_hour < 10).mean():.0%}")

In [ ]:
give_solution("2.5", show=False)  # show=True shows the solution

In [ ]:
both = pd.DataFrame({"No Calibration": no_calibration, "With Calibration": with_calibration})
ax = both.plot(color=["#E69F00", "#009E73"], style=["--", "-"])
ax.axhline(10, color="red", linewidth=0.8, linestyle=":", label="NFR Threshold (10%)")
ax.axhline(7.86, color="#338ec1", linewidth=0.8, linestyle="-.", label="FootPrinter (7.86%)")
ax.set(xlabel="Time [UTC]", ylabel="MAPE [%]", ylim=(0, None))
legend_above(ax)

In [ ]:
ask_question("2.6")

In [ ]:
give_solution("2.6", show=False)  # show=True shows the solution

In [ ]:
ask_question("2.7")

In [ ]:
worse = with_calibration > no_calibration
print(f"Calibration is worse than no calibration in {worse.sum()} of {len(worse)} hours.")

In [ ]:
give_solution("2.7", show=False)  # show=True shows the solution

In [ ]:
ask_question("2.8")

In [ ]:
last_24h = errors.rolling("24h").mean().resample("1h").last()  # MAPE of each r over the last 24 hours
with_24h = self_calibrate(hourly, last_24h.idxmin(axis=1))
print(f"MAPE {with_24h.mean():.1f}%, worse than no calibration in {(with_24h > no_calibration).sum()} hours")

In [ ]:
give_solution("2.8", show=False)  # show=True shows the solution

In [ ]:
ask_question("2.9")

In [ ]:
best_r.value_counts().sort_index()

In [ ]:
give_solution("2.9", show=False)  # show=True shows the solution

In [ ]:
ask_question("2.10")

In [ ]:
sweep_wide = simulate_many("r", R_VALUES + [10, 11, 12], until=TRACE_END)  # r = 3 to 9 come from the cache
errors_wide = pd.DataFrame({r: error(run) for r, run in sweep_wide.items()})
hourly_wide = errors_wide.resample("1h").mean()
best_r_wide = hourly_wide.idxmin(axis=1)
with_wide = self_calibrate(hourly_wide, best_r_wide)
print(f"r from 3 to 12: MAPE {with_wide.mean():.1f}%")
best_r_wide.value_counts().sort_index()

In [ ]:
give_solution("2.10", show=False)  # show=True shows the solution

## Calibration in the Live Twin

*Optional, requires Docker.* The same grid search, in `resources/configs/live_twin_calibration.yaml`:

```yaml
calibrator:
  calibrated_property: "cpuPowerModel.calibrationFactor"  # r in the power model
  min_value: 3
  max_value: 9
  linspace_points: 7  # r = 3, 4, ..., 9
  max_parallel_workers: 4
  mape_window_minutes: 60
```

Start it with `make up config=tutorial/resources/configs/live_twin_calibration.yaml` and follow it with
`make logs-calibrator`: after each grid search, the MAPE of each candidate, the best value
(`Best: cpuPowerModel.calibrationFactor=...`), and `Published topology with ...`. More in notebook 4.

*Next:* [3.co2_migration.ipynb](3.co2_migration.ipynb), CO2-aware workload migration.